# 01 · Receber lote
Simula a chegada diária de dados: copia **o próximo** CSV da pasta `chegada/` para `pedidos/`. Na vida real, quem deposita o arquivo é o sistema de origem (PDV, ERP, API).

Esta é a **1ª tarefa do job** (nome da tarefa: `receber_lote`). Ela publica dois *task values*:

| chave | valor |
|---|---|
| `novo_lote` | `1` se copiou um arquivo, `0` se não havia nada novo |
| `nome_lote` | nome do arquivo copiado |

In [ ]:
BASE    = "/Volumes/workspace/aula_jobs/arquivos"
CHEGADA = f"{BASE}/chegada/"
PEDIDOS = f"{BASE}/pedidos/"

disponiveis  = sorted(f.name for f in dbutils.fs.ls(CHEGADA) if f.name.endswith(".csv"))
ja_entregues = {f.name for f in dbutils.fs.ls(PEDIDOS)}
pendentes    = [a for a in disponiveis if a not in ja_entregues]

if pendentes:
    nome_lote = pendentes[0]
    dbutils.fs.cp(CHEGADA + nome_lote, PEDIDOS + nome_lote)
    novo_lote = 1
    print(f"📦 Lote recebido: {nome_lote}  (restam {len(pendentes) - 1} na fila)")
else:
    nome_lote = ""
    novo_lote = 0
    print("😴 Nenhum lote novo em chegada/.")

### Publicar valores para as próximas tarefas
Dentro do job, a tarefa *If/else* lê `{{tasks.receber_lote.values.novo_lote}}`. Fora do job não há para quem publicar — tudo bem rodar esta célula mesmo assim.

In [ ]:
try:
    dbutils.jobs.taskValues.set(key="novo_lote", value=novo_lote)
    dbutils.jobs.taskValues.set(key="nome_lote", value=nome_lote)
    print("Task values publicados:", {"novo_lote": novo_lote, "nome_lote": nome_lote})
except Exception as e:
    print("(Fora de um job: task values não publicados)", e)